## Samenvatting

Een operationeel logistiek team plant een gerandomiseerde veldproef die drie chauffeursroutestrategieën vergelijkt (statische basislijn, dynamische herroutering, AI-geoptimaliseerd) over drie depotregio's, met de gemiddelde bezorgvertraging (in minuten) als respons. PROC GLMPOWER neemt een *voorbeeld*dataset van veronderstelde celgemiddelden en berekent het totale aantal chauffeurs dat nodig is om strategie-effecten te detecteren bij 80% en 90% onderscheidingsvermogen, en brengt vervolgens in kaart hoe het haalbare onderscheidingsvermogen afneemt naarmate de variabiliteit tussen routes toeneemt.

# Een chauffeursroute-veldproef dimensioneren met PROC GLMPOWER

## Samenvatting

Een operationeel logistiek team staat op het punt een gerandomiseerde veldproef te lanceren die drie chauffeursroutestrategieën vergelijkt -- een **Static**-basislijn, een **Dynamic**-herrouteringssysteem en een **AI-Optimized**-planner -- ingezet over drie depotregio's (Northeast, Midwest, West). De respons is de gemiddelde **bezorgvertraging in minuten**. Voordat het team vlootcapaciteit aan de proef toewijst, moet het antwoorden: *hoeveel chauffeurs hebben we nodig om de operationele verbetering die we verwachten betrouwbaar te detecteren?*

Deze notebook gebruikt **PROC GLMPOWER** om prospectieve power- en steekproefgrootte-analyse uit te voeren voor het algemene lineaire model achter de proef. Uitgaande van een *voorbeeld*dataset van veronderstelde celgemiddelden: (1) berekent het de totale instroom die nodig is om 80% en 90% onderscheidingsvermogen te bereiken voor de omnibus-strategie- en regio-effecten, (2) brengt het in kaart hoe het haalbare onderscheidingsvermogen afneemt naarmate de variabiliteit tussen routes toeneemt, en (3) produceert het een vermogenscurve ter ondersteuning van de instroombeslissing.

> **Belangrijkste inzicht:** Bij een plausibele foutstandaarddeviatie van 8 minuten leveren ongeveer **63 chauffeurs** 80% onderscheidingsvermogen en **83 chauffeurs** 90% onderscheidingsvermogen op voor het detecteren van routeringsstrategie-effecten -- maar als de vertragingsvariabiliteit dichter bij 10 minuten ligt, blijft zelfs bij 90 ingezette chauffeurs het onderscheidingsvermogen onder 70%, wat het belang onderstreept van strakke, goed geïnstrumenteerde routes.

---
## 1. Het voorbeeldontwerp opbouwen

De eerste stap codeert de proefopzet en de door het team veronderstelde gemiddelde vertraging voor elke combinatie van routeringsstrategie x depotregio. We leggen een random seed vast en voegen een verwaarloosbare jitter toe zodat de gemiddelden realistisch ogen terwijl de gebalanceerde 3x3-structuur behouden blijft. Het `cell_n`-gewicht van 1 in elke cel vertelt GLMPOWER dat het ontwerp gebalanceerd is.

In [1]:
/* ================================================================
   Genereer de voorbeelddataset van veronderstelde gemiddelde
   vertragingen. Eén rij per routeringsstrategie x depotregio
   ontwerpcel.
   ================================================================ */
GEGEVENS routing_trial;
   CALL streaminit(20260531);
   LENGTE routing_strategy $8 depot_region $9;
   REEKS strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   REEKS region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   REEKS smean[3]     _temporary_ (18.0 14.5 11.0);   /* veronderstelde strategiegemiddelden */
   REEKS radj[3]      _temporary_ (1.5  0.0 -1.0);    /* regionale correcties (min)  */
   DOE i = 1 TOT 3;
      DOE j = 1 TOT 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         UITVOER;
      EINDE;
   EINDE;
   VERWIJDEREN i j;
   label routing_strategy="Routeringsstrategie" depot_region="Depotregio"
         mean_delay_min="Gemiddelde bezorgvertraging (min)" cell_n="Celgewicht";
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=routing_trial label noobs;
   VARIABELE routing_strategy depot_region mean_delay_min cell_n;
   TITEL "Voorbeeldcelgemiddelden: veronderstelde bezorgvertraging (minuten)";
UITVOEREN;

                           Voorbeeldcelgemiddelden: veronderstelde bezorgvertraging (minuten)                           

Routeringsstrategie  Depotregio  Gemiddelde bezorgvertraging (min)  Celgewicht
Static               Northeast                        19.687507651           1
Static               Midwest                         17.9871067398           1
Static               West                            16.8240210086           1
Dynamic              Northeast                       15.9537535365           1
Dynamic              Midwest                         14.4415169858           1
Dynamic              West                            12.8636389493           1
AIOpt                Northeast                       12.6143811724           1
AIOpt                Midwest                          10.893885771           1
AIOpt                West                              9.635351403           1




NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Steekproefgrootte voor het omnibusontwerp

Met het ontwerp in handen vragen we GLMPOWER om **de totale steekproefgrootte op te lossen** (`NTOTAL = .`) bij 80% en 90% onderscheidingsvermogen. Het `MODEL`-statement specificeert een tweewegindeling met interactie (`routing_strategy*depot_region`), `WEIGHT cell_n` declareert de gebalanceerde profielgewichten, en `STDDEV = 8` is de veronderstelde root-mean-square-fout van de bezorgvertraging. `NFRACTIONAL` laat de procedure exacte fractionele aantallen rapporteren voordat er wordt afgerond.

We registreren ook vooraf drie geplande `CONTRAST`-vergelijkingen -- AI-Opt versus Static, Dynamic versus Static, en elke herroutering versus Static -- die de operationeel betekenisvolle hypothesen documenteren waarvoor de proef is opgezet.

In [2]:
/* ================================================================
   Los op voor het totale aantal chauffeurs dat nodig is om 80% en
   90% onderscheidingsvermogen te bereiken voor de effecten van
   routeringsstrategie en depotregio.
   ================================================================ */
PROCEDURE glmpower GEGEVENS=routing_trial;
   KLASSE routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   GEWICHT cell_n;
   label routing_strategy="Routeringsstrategie" depot_region="Depotregio" mean_delay_min="Gemiddelde bezorgvertraging (min)";
   CONTRAST "AI-Opt versus Static"      routing_strategy -1  0  1;
   CONTRAST "Dynamic versus Static"     routing_strategy -1  1  0;
   CONTRAST "Elke herroutering versus Static" routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   TITEL "Steekproefgrootte om effecten van routeringsstrategie op vertraging te detecteren";
UITVOEREN;

                           Voorbeeldcelgemiddelden: veronderstelde bezorgvertraging (minuten)                           

The GLMPOWER Procedure


               Fixed Scenario Elements               

Item                Value                            
------------------  ---------------------------------
Dependent Variable  Gemiddelde bezorgvertraging (min)
Source              Routeringsstrategie              
Source              Depotregio                       
Source              Routeringsstrategie*Depotregio   

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Gevoeligheid van onderscheidingsvermogen voor variabiliteit en proefomvang

Het antwoord op de steekproefgrootte hangt af van de veronderstelde foutstandaarddeviatie, die vooraf zelden precies bekend is. Hier keren we de vraag om: we **leggen** verschillende kandidaat-instroomtotalen vast (`NTOTAL = 45 90 180`) en **lossen op voor het behaalde onderscheidingsvermogen** (`POWER = .`) over een raster van plausibele standaarddeviaties van de vertraging (6, 8 en 10 minuten). De resulterende tabel is een gevoeligheidskaart -- ze toont hoe robuust elk instroomplan is als de variabiliteit van routes in de praktijk hoger uitvalt dan gehoopt.

In [3]:
/* ================================================================
   Gevoeligheidsraster: behaald onderscheidingsvermogen over
   kandidaat-proefomvangen en plausibele standaarddeviaties van
   de vertraging.
   ================================================================ */
PROCEDURE glmpower GEGEVENS=routing_trial;
   KLASSE routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   GEWICHT cell_n;
   label routing_strategy="Routeringsstrategie" depot_region="Depotregio" mean_delay_min="Gemiddelde bezorgvertraging (min)";
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   TITEL "Behaald onderscheidingsvermogen bij scenario's voor variabiliteit en proefomvang";
UITVOEREN;

                           Voorbeeldcelgemiddelden: veronderstelde bezorgvertraging (minuten)                           

The GLMPOWER Procedure


               Fixed Scenario Elements               

Item                Value                            
------------------  ---------------------------------
Dependent Variable  Gemiddelde bezorgvertraging (min)
Source              Routeringsstrategie              
Source              Depotregio                       

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Vermogenscurve voor de instroombeslissing

Tot slot brengen we een **vermogenscurve** in kaart -- behaald onderscheidingsvermogen naarmate de totale instroom groeit van 30 naar 270 chauffeurs in stappen van 30 -- voor het strategie-plus-regiomodel bij de verwachte standaarddeviatie van 8 minuten. Het oplossen van `POWER = .` over dat instroomraster produceert de curve als een getabelleerde reeks `N Total` versus `Power`, zodat we kunnen aflezen bij welke instroom elk van de conventionele doelen van 80% en 90% wordt gehaald.

In [4]:
/* ================================================================
   Vermogenscurve: behaald onderscheidingsvermogen versus totaal
   aantal ingezette chauffeurs, doorlopen van 30 tot 270 in stappen
   van 30 bij de verwachte standaarddeviatie van 8 min.
   ================================================================ */
PROCEDURE glmpower GEGEVENS=routing_trial;
   KLASSE routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   GEWICHT cell_n;
   label routing_strategy="Routeringsstrategie" depot_region="Depotregio" mean_delay_min="Gemiddelde bezorgvertraging (min)";
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   TITEL "Onderscheidingsvermogencurve: behaald vermogen versus totaal aantal ingezette chauffeurs";
UITVOEREN;

                           Voorbeeldcelgemiddelden: veronderstelde bezorgvertraging (minuten)                           

The GLMPOWER Procedure


               Fixed Scenario Elements               

Item                Value                            
------------------  ---------------------------------
Dependent Variable  Gemiddelde bezorgvertraging (min)
Source              Routeringsstrategie              
Source              Depotregio                       

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Interpretatie en aanbeveling

De analyse geeft het operationele team een verdedigbaar instroomplan:

- **Basisdimensionering (Sectie 2).** Uitgaande van een residuele standaarddeviatie van 8 minuten bereikt het volledige tweewegmodel (strategie, regio en hun interactie) **80% onderscheidingsvermogen bij 63 chauffeurs** en **90% onderscheidingsvermogen bij 83 chauffeurs**. Naar boven afgerond voor uitval komt een instroomdoel van rond de **90 chauffeurs** ruim boven de drempel van 90% uit.

- **Gevoeligheid is belangrijk (Sectie 3).** Het onderscheidingsvermogen is zeer gevoelig voor de vertragingsvariabiliteit. Bij 90 chauffeurs daalt het behaalde vermogen van ~99% (SD=6) naar ~87% (SD=8) naar ~68% (SD=10). Een pilot met 45 chauffeurs is alleen toereikend als de variabiliteit laag is (~81% vermogen bij SD=6) en is ernstig onderbemeten (~37%) als SD tot 10 oploopt. De praktische implicatie: investeren in consistente, goed geïnstrumenteerde routes om de variabiliteit laag te houden is even waardevol als het toevoegen van chauffeurs.

- **De vermogenscurve (Sectie 4).** Voor het strategie-plus-regiomodel (zonder interactieterm, de lens die is gebruikt voor de gevoeligheidsanalyse) stijgt het behaalde vermogen van 0,37 bij 30 chauffeurs naar 0,69 bij 60, 0,87 bij 90 en 0,95 bij 120, en vlakt het af boven 0,99 rond 180. Als we de curve tegen de doelen aflezen, wordt 80% vermogen bereikt bij ongeveer 77 chauffeurs en 90% bij ongeveer 99 -- iets hoger dan de dimensionering van het volledige model in Sectie 2, omdat het weglaten van de interactieterm hetzelfde effect over minder modelvrijheidsgraden spreidt.

**Aanbeveling:** Zet ongeveer **90 chauffeurs** in (30 per routeringsstrategie, gebalanceerd over de drie depotregio's). Dit haalt 90% onderscheidingsvermogen voor het volledige model bij de verwachte standaarddeviatie van 8 minuten, houdt ~87% vermogen aan zelfs op de conservatievere gereduceerde-modelcurve, en houdt de proef klein genoeg om binnen één operationeel kwartaal uit te voeren.

*Opmerking:* GLMPOWER analyseert het veronderstelde *ontwerp*, niet waargenomen uitkomsten -- dus de geloofwaardigheid van deze cijfers berust op de veronderstelde gemiddelden en standaarddeviatie. Teams moeten de dimensionering herzien zodra een korte pilot een empirische schatting van de bezorgvertragingsvariabiliteit oplevert.